# Spam Classifier

Spam classifier built with the spamassasin dataset
[Source](https://spamassassin.apache.org/old/publiccorpus/)

# Loading the libraries

In [ ]:
import pathlib
import pandas as pd
import email
import re
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

# Loading the Dataset

In [49]:
def load_email(path):
    """ Better way to load the email data """
    try:
        with open(path, "rb") as file:
            return email.parser.BytesParser(policy=email.policy.default).parse(file)
    except Exception as e:
        print(f"Error loading email from {path}: {e}")
        return ""

def html_to_text(html):
    """ Convert HTML content to plain text """
    soup = BeautifulSoup(html, "html.parser")
    
    for tag in soup(["script", "style"]):
        tag.decompose()
    
    # Remove whitespace
    text = soup.get_text(separator=" ", strip=True)

    lines = (line.strip() for line in text.splitlines())
    chunks = (phrase.strip() for line in lines for phrase in line.split(" "))
    return "\n".join(chunk for chunk in chunks if chunk)

def clean_text(text):
    """ Clean the text by removing extra whitespace and non-alphanumeric characters """
    text = text.lower()
    
    # Replacing URLs
    url_pattern = r"(https?://\S+|www\.\S+)"
    text = re.sub(url_pattern, " URL ", text)

    # Replacing numbers
    number_pattern = r"\d+(\.\d+)?"
    text = re.sub(number_pattern, " NUMBER ", text)

    # Remove extra whitespaces
    text = re.sub(r"\s+", ' ', text).strip()

    return text

def safe_decode(part):
    """ Safely decode email content """
    try:
        payload = part.get_payload(decode=True)

        if payload is None:
            return ""
        
        charset = part.get_content_charset()

        if not charset or charset.lower() == "default":
            charset = "utf-8"
        
        return payload.decode(charset, errors="ignore")
    except Exception as e:
        print(f"Fallback: decoding content: {e}, trying with latin-1")
        return payload.decode("latin-1", errors="ignore")

    
def get_email_content(mail):
    """ Get the email content as text """

    # If it's not a multipart email
    if not mail.is_multipart():
         return clean_text(safe_decode(mail).strip())
    
    # If it's a multipart email (HTML + Text)
    for part in mail.walk():
        content_type = part.get_content_type()
        content_disposition = str(part.get("Content-Disposition"))

        # Skip attachments
        if "attachment" in content_disposition:
             continue
        
        # Get the email body 
        if content_type == "text/plain":
             return clean_text(safe_decode(part).strip())
        elif content_type == "text/html":
             return clean_text(html_to_text(safe_decode(part)))
        
    return "" # return an empty string if not found


def load_dataset(data_dir):
    """Loading the dataset"""
    data = []

    for file_path in pathlib.Path(data_dir).rglob("*"):
        if file_path.is_file():
                mail = load_email(file_path)
                # True if spam, False if ham
                label = True if "spam" in file_path.parts else False
                mail_content = safe_decode(load_email(file_path))
                data.append({"content": mail_content, "label": label})

    return pd.DataFrame(data)

base_path = pathlib.Path.cwd().parent / "dataset"

# Get the email content as text
mail = load_dataset(base_path)

X = mail["content"]
y = mail["label"]

Fallback: decoding content: unknown encoding: chinesebig5, trying with latin-1
Fallback: decoding content: unknown encoding: default_charset, trying with latin-1
Fallback: decoding content: unknown encoding: default_charset, trying with latin-1
Fallback: decoding content: unknown encoding: default_charset, trying with latin-1
Fallback: decoding content: unknown encoding: default_charset, trying with latin-1
Fallback: decoding content: unknown encoding: default_charset, trying with latin-1
Fallback: decoding content: unknown encoding: default_charset, trying with latin-1
Fallback: decoding content: unknown encoding: gb2312_charset, trying with latin-1
Fallback: decoding content: unknown encoding: unknown-8bit, trying with latin-1
Fallback: decoding content: unknown encoding: default_charset, trying with latin-1
Fallback: decoding content: unknown encoding: default_charset, trying with latin-1
Fallback: decoding content: unknown encoding: default_charset, trying with latin-1


# Seperating into test, train sets

In [50]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)} {sum(y_train)} ")
print(f"Test set size: {len(X_test)} {sum(y_test)} ")

Training set size: 3520 1541 
Test set size: 880 358 


# Vectorization

Converting each sentence into a vector based on a given set of words

In [51]:
from sklearn.feature_extraction.text import CountVectorizer

# Initialization
vectorizer = CountVectorizer(binary=True)

# Fit and transform the training data
X_train_vectorized = vectorizer.fit_transform(X_train)

# Transform the test data (don't fit)
X_test_vectorized = vectorizer.transform(X_test)

# Custom transformer for preparing a clean text

In [52]:
class EmailToCleanTextTransformer(BaseEstimator, TransformerMixin):
    """Custom transformer to convert email content to clean text"""

    def __init__(self, filter_html=True, clean_text=True):
        self.filter_html = filter_html
        self.clean_text = clean_text

    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        X_transformed = []

        for email_content in X:
            if self.filter_html:
                email_content = html_to_text(email_content)
            if self.clean_text:
                email_content = clean_text(email_content)
            X_transformed.append(email_content)
        
        return X_transformed

# Preparing a pipeline

In [54]:
spam_pipeline = Pipeline([
    ("email_to_text", EmailToCleanTextTransformer()),
    ("vectorizer", CountVectorizer(binary=True)),
    ("classifier", LogisticRegression(solver="liblinear", max_iter=1000))
])

spam_pipeline.fit(X_train, y_train)

y_pred = spam_pipeline.predict(X_test)

# Evaluation

In [ ]:
print(confusion_matrix(y_test, y_pred))